## FastAPI 前置知识
- 熟悉 Python 基础知识（类型提示、装饰器、生成器、上下文管理器、类与继承）
- 了解异步编程概念(event loop、async/await、async def vs def)
- 掌握基本的 HTTP 协议知识（请求方法、状态码、请求组成、响应处理、RESTful API）

### 一、HTTP协议知识

#### 1. 请求方法
- GET 用于获取资源，没有请求体（浏览器请求）
- POST 创建或提交处理，通常有请求体
- PUT 用于整体更新
- PATCH 用于部分更新
- DELETE 用于删除

FastAPI 用 `@app.get()`、`@app.post()` 等装饰器直接绑定方法

#### 2. 请求的组成

一个 HTTP 请求由请求方法、URL（路径参数、查询参数）、请求体、请求头组成。
- 路径参数：URL 路径中 `{}` 的变量，如 `/items/{item_id}`；
- 查询参数：URL 中 `?` 之后的部分，如 `?skip=0&limit=10`；
- 请求体：通常用于 POST/PUT/PATCH，承载 JSON 数据
- 请求头：携带元数据（`Content-Type`、`Authorization` 等），FastAPI 通过 `Header` 参数注入

```
@app.get("/items/{item_id}")
async def read_item(item_id: int, skip: int, limit: int):  # 类型提示触发自动转换和校验
    ...

```

```
https://api.example.com:8443/items/42?skip=0&limit=10
└─┬─┘   └──────┬───────┘└┬─┘└───┬───┘└──────┬────────┘
scheme      host        port  path        query
```

真正进入 FastAPI 的是 `path` 和 `query`。

FastAPI 的判定规则可以概括为：
- 路径模板里出现的名字 → `路径参数`；简单类型且不在路径中 → `查询参数`；Pydantic 模型 → `请求体`；
- `Header/Cookie/Form/File` 显式声明 → 对应位置。

理解这条规则，就不会对“为什么这个参数跑到 query 里去了”感到困惑。

请求头 `Content-Type` 告诉服务端请求体是什么格式

| Content-Type | 典型场景 | FastAPI 声明方式 |
| :--- | :--- | :--- |
| `application/json` | 大多数 API | Pydantic 模型参数或 `Body()` |
| `application/x-www-form-urlencoded` | HTML 表单 | `Form()` |
| `multipart/form-data` | 文件上传 + 表单 | `File()` / `UploadFile` |
| `text/plain` | 纯文本 | `Body(media_type="text/plain")` 或读 `Request.body()` |
| `application/octet-stream` | 二进制流 | 读 `Request.stream()` 或 `UploadFile` |


#### 4. 请求与响应的结构对称性

HTTP 响应同样由状态码、响应头、响应体组成。
- FastAPI 通过 `response_model` 控制响应体结构，
- 通过 `status_code` 参数或 `JSONResponse` 定制状态码和响应头。

理解请求与响应的这种对称结构，有助于正确使用 FastAPI 的 `Request`、`Response`、`JSONResponse` 等对象

#### 状态码
- 1xx 信息性
- 2xx 成功
- 3xx 重定向
- 4xx 客户端错误
- 5xx 服务端错误

至少需要熟悉以下语义，并知道它们在 FastAPI 中如何体现：
- 200 OK：成功；
- 201 Created：创建成功（POST 创建资源后常用）；
- 204 No Content：成功但无返回体（如 DELETE）；
- 400 Bad Request：客户端请求语义错误；
- 401 / 403：401-未认证 / 403-已认证但无权限；
- 404 Not Found：资源不存在；
- 405 Method Not Allowed：路径存在但方法不支持
- 422 Unprocessable Entity：FastAPI 中请求体验证失败时自动返回的状态码，这是 FastAPI 开发者最常打交道的非 2xx 码之一；
- 429 Too Many Requests：限流。FastAPI 不内置限流，需中间件或依赖实现
- 500 Internal Server Error：服务端未处理异常。